# 01 Data Collection

## Objective
Collect and prepare the raw network data required to model the London Underground as a graph.

## Data Sources
- TfL StopPoint API for station metadata
- TfL Line Route Sequence API for line connections
- Local raw CSV files for later operational analysis

## Inputs
- TfL API responses
- `data/raw/service-operated.csv`
- `data/raw/kilometres-operated.csv`
- `data/raw/excess-journey-time.csv`
- `data/raw/orr_station_usage.csv`
- `data/raw/trains_planned.csv`

## Outputs
- `data/processed/tfl_stations.csv`
- `data/processed/tube_connections.csv`

## Why this step matters
This notebook creates the base station and route datasets used in all downstream graph analysis.

In [1]:
import pandas as pd
import requests
from pathlib import Path

In [2]:
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())

RAW_DIR: C:\Users\Saisha\OneDrive\Desktop\transport-network-failure-analysis\data\raw
PROCESSED_DIR: C:\Users\Saisha\OneDrive\Desktop\transport-network-failure-analysis\data\processed


In [3]:
print("Available raw files:")
for f in RAW_DIR.iterdir():
    print("-", f.name)

Available raw files:
- excess-journey-time.csv
- kilometres-operated.csv
- orr_station_usage.csv
- service-operated.csv
- trains_planned.csv


In [4]:
# Optional quick checks that raw files exist
required_raw = [
    "service-operated.csv",
    "kilometres-operated.csv",
    "excess-journey-time.csv",
    "orr_station_usage.csv",
    "trains_planned.csv",
]

missing_raw = [f for f in required_raw if not (RAW_DIR / f).exists()]
print("Missing raw files:", missing_raw if missing_raw else "None")

Missing raw files: None


## Step 1: Collect station metadata from the TfL API

This step retrieves London Underground station information such as station ID, name, latitude, longitude, and zone.

In [5]:
url = "https://api.tfl.gov.uk/StopPoint/Mode/tube"
response = requests.get(url)

print("Status code:", response.status_code)
tube_json = response.json()

print(type(tube_json))
print(tube_json.keys())

Status code: 200
<class 'dict'>
dict_keys(['$type', 'stopPoints', 'pageSize', 'total', 'page'])


In [6]:
stations_list = []

for item in tube_json.get("stopPoints", []):
    stations_list.append({
        "station_id": item.get("id"),
        "station_name": item.get("commonName"),
        "lat": item.get("lat"),
        "lon": item.get("lon"),
        "zone": item.get("zone")
    })

stations_df = pd.DataFrame(stations_list)
print(stations_df.shape)
stations_df.head()

(1751, 5)


,station_id,station_name,lat,lon,zone
0,0400ZZLUAMS0,Amersham Underground Station,51.674206,-0.607362,None
1,0400ZZLUCAL0,Chalfont & Latimer Underground Station,51.667915,-0.560616,None
2,0400ZZLUCAL1,Chalfont & Latimer Underground Station,51.668122,-0.560624,None
3,0400ZZLUCSM0,Chesham Underground Station,51.705227,-0.611113,None
4,2100ZZLUCXY0,Croxley Underground Station,51.647069,-0.441746,None


In [7]:
stations_df.to_csv(PROCESSED_DIR / "tfl_stations.csv", index=False)
print("Saved tfl_stations.csv")

Saved tfl_stations.csv


In [8]:
lines = [
    "bakerloo", "central", "circle", "district",
    "hammersmith-city", "jubilee", "metropolitan",
    "northern", "piccadilly", "victoria", "waterloo-city"
]

## Step 2: Collect line route connections from the TfL API

This step retrieves route sequences for Underground lines and converts them into station-to-station connection pairs.

In [9]:
all_connections = []

for line in lines:
    route_url = f"https://api.tfl.gov.uk/Line/{line}/Route/Sequence/all"
    r = requests.get(route_url)

    print(f"{line}: {r.status_code}")

    if r.status_code == 200:
        data = r.json()

        for line_string in data.get("orderedLineRoutes", []):
            naptan_ids = line_string.get("naptanIds", [])

            for i in range(len(naptan_ids) - 1):
                all_connections.append({
                    "line": line,
                    "from_id": naptan_ids[i],
                    "to_id": naptan_ids[i + 1]
                })

bakerloo: 200
central: 200
circle: 200
district: 200
hammersmith-city: 200
jubilee: 200
metropolitan: 200
northern: 200
piccadilly: 200
victoria: 200
waterloo-city: 200


In [10]:
connections_df = pd.DataFrame(all_connections).drop_duplicates()
print(connections_df.shape)
connections_df.head()

(754, 3)


,line,from_id,to_id
0,bakerloo,940GZZLUEAC,940GZZLULBN
1,bakerloo,940GZZLULBN,940GZZLUWLO
2,bakerloo,940GZZLUWLO,940GZZLUEMB
3,bakerloo,940GZZLUEMB,940GZZLUCHX
4,bakerloo,940GZZLUCHX,940GZZLUPCC


## Step 3: Save processed network source files

The cleaned station and connection outputs are saved to the `data/processed` folder for use in later notebooks.

In [11]:
connections_df.to_csv(PROCESSED_DIR / "tube_connections.csv", index=False)
print("Saved tube_connections.csv")

Saved tube_connections.csv
